# 112 — CPTAC-3: Upstream RNA-seq QC-Source Provenance Pilot

## Scientific and operational purpose

**Phase 1 acquisition/metadata-audit complement to the executed Notebook 111.** Verify the *source of the six quantitative QC fields* observed in twelve H-pilot STAR Counts workflow inputs and test whether the upstream input-file records carry a **direct, exact case/aliquot association** that can be reconciled with the corresponding STAR output files. The 12 STAR output file UUIDs and their recorded workflow input IDs were captured in the **immutable-on-rerun H metadata snapshot** from Notebook 800; H is the *discovery provenance*, not a source of final sample eligibility.

This notebook is *not* a secondary sequencing data download, cohort-wide QC survey, case-selection algorithm, or external program-validation analysis. It does **not** read BAM/FASTQ payloads, download any files, read the 1,082 STAR Count payloads, rerun Notebook 111, expand to 1,082/649 files, compute expression scores, define QC thresholds, or alter any frozen Phase 2–7 artifact.

### Falsifiable, bounded questions

1. Do the **upstream input-file UUIDs** embedded in H correspond to distinct, retrievable GDC \`/files\` file records?
2. Does each upstream record have exactly one **direct \`associated_entities\` aliquot** with the *same case and aliquot UUID* as the previously verified STAR output file? Case membership by itself is not enough.
3. Are the six quantitative source-file fields available in those **direct** records, and do they agree with the previously observed H \`analysis.input_files\` projection? The two projections can contain the *same provider-managed metadata*; agreement is a **provenance consistency check, not an independent measurement or QC validation**.
4. Are field definitions/read-or-fragment denominators and library/pipeline compatibility documented sufficiently for a later prospective analytic QC policy? Absence of such confirmation is a **BLOCKER**, not a QC failure or exclusion for a person.

**Fixed scope**: twelve STAR outputs; at most twelve unique upstream input-file UUIDs; exactly one GDC metadata-only query, snapshot versioned and fingerprinted. A GDC retrieval failure stops the notebook; missing exact input-file records or direct associations are explicitly classified instead of interpreted as provider-wide absence. No automatic expansion or source rescue is allowed.

GDC reference: [aligned-read summary metrics](https://docs.gdc.cancer.gov/Data/Bioinformatics_Pipelines/Aligned_reads_summary_metrics/), [available API fields](https://docs.gdc.cancer.gov/API/Users_Guide/Appendix_A_Available_Fields/), [RNA expression workflow](https://docs.gdc.cancer.gov/Data/Bioinformatics_Pipelines/Expression_mRNA_Pipeline/).

**Execution**: top-to-bottom in the existing Python 3.11.8 \`.venv\`. The already executed H JSON snapshot and 111 audit tables must be present locally. Do not reproduce H by calling Notebook 800. On a new machine without those snapshots, stop and recover the authenticated handoffs before continuing.

In [ ]:
# =============================================================================
# Imports and fixed independent-acquisition source scope
# =============================================================================

import hashlib
import json
import math
import platform
import time
from datetime import datetime, timezone

import pandas as pd
import requests

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path

ROOT = Paths.metadata
AUDIT_111_PATH = ROOT / "111_cptac3_acquisition_audit_summary.json"
FILE_INDEX_111_PATH = ROOT / "111_cptac3_file_index.csv"
H_SNAPSHOT_PATH = ROOT / "800_preflight_h_gdc_qc_source_pilot.json"
H_READINESS_PATH = ROOT / "800_preflight_a_readiness.json"

SNAPSHOT_112_PATH = ROOT / "112_cptac3_qc_upstream_direct_file_snapshot.json"
OUTPUT_112_PATHS = {
    "source_identities": ROOT / "112_cptac3_qc_input_file_provenance.csv",
    "metric_comparison": ROOT / "112_cptac3_qc_input_metric_reconciliation.csv",
    "source_summary": ROOT / "112_cptac3_qc_source_feasibility.csv",
    "handoff": ROOT / "112_cptac3_qc_provenance_handoff.json",
}

QC_FIELDS = (
    "total_reads",
    "proportion_reads_mapped",
    "proportion_reads_duplicated",
    "average_base_quality",
    "average_read_length",
    "average_insert_size",
)

for name, path in {
    "111_audit": AUDIT_111_PATH,
    "111_file_index": FILE_INDEX_111_PATH,
    "800_H_snapshot": H_SNAPSHOT_PATH,
    "800_H_readiness": H_READINESS_PATH,
}.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"{name} is missing: {project_relative_path(path)}. "
            "Stop and recover the prior immutable handoff; do not refetch or reconstruct."
        )
print("Notebook 112 source handoffs exist; no STAR/BAM/FASTQ payload reads")

In [ ]:
# =============================================================================
# Reconcile 111 and H identities and cryptographic upstream dependencies
# =============================================================================

with AUDIT_111_PATH.open("r", encoding="utf-8") as h:
    audit_111 = json.load(h)
with H_READINESS_PATH.open("r", encoding="utf-8") as h:
    readiness_h = json.load(h)
with H_SNAPSHOT_PATH.open("r", encoding="utf-8") as h:
    snapshot_h = json.load(h)

h_receipt = (readiness_h.get("inputs") or {}).get(
    "gdc_h_targeted_qc_source_pilot"
)
if not isinstance(h_receipt, dict):
    raise ValueError("H readiness has no exact-query snapshot fingerprint")
if (
    calculate_sha256(H_SNAPSHOT_PATH) != h_receipt.get("sha256")
    or snapshot_h.get("query_sha256") != h_receipt.get("query_sha256")
    or readiness_h.get("rna_preflight_h", {}).get("pilot_star_files") != 12
    or snapshot_h.get("notebook111_rna_metadata_sha256")
    != audit_111["source_fingerprints"]["rnaseq"]["metadata"]["sha256"]
    or (readiness_h.get("upstream_111_snapshot") or {}).get(
        "rnaseq", {}
    ).get("metadata", {}).get("sha256")
    != audit_111["source_fingerprints"]["rnaseq"]["metadata"]["sha256"]
    or not audit_111.get("full_payload_md5_verified", False)
):
    raise ValueError("111/H audit handoff fingerprints, scopes or status disagree")

idx_info = (audit_111.get("audit_artifacts") or {}).get("file_index")
if not isinstance(idx_info, dict) or (
    calculate_sha256(FILE_INDEX_111_PATH) != idx_info.get("sha256")
):
    raise ValueError("111 file-index SHA-256 does not match its own audit summary")

index_111 = pd.read_csv(FILE_INDEX_111_PATH, dtype="string")
expected = {"file_id", "filename", "size_bytes", "provider_md5", "modality",
            "case_uuid", "sample_uuid", "aliquot_uuid", "mapping_status",
            "sample_type"}
if not expected.issubset(index_111.columns):
    raise KeyError(f"Incomplete 111 file index: {expected-set(index_111.columns)}")
if index_111.duplicated(["modality", "file_id"]).any():
    raise ValueError("111 source file index contains duplicate file UUID")
idx_rna = index_111.loc[index_111["modality"].eq("rnaseq")].copy()
hits_h = snapshot_h.get("hits")
if (
    snapshot_h.get("role")
    != "PREFLIGHT_H_METADATA_SOURCE_FEASIBILITY_PILOT_ONLY"
    or not isinstance(hits_h, list)
    or len(hits_h) != 12
    or len({x.get("file_id") or x.get("id") for x in hits_h}) != 12
):
    raise ValueError("H pilot is not exactly 12 immutable STAR records")
print("111 source index: verified", len(idx_rna), "RNA source files")
print("H source snapshot: verified 12 exact STAR output records")
print("H snapshot SHA-256:", calculate_sha256(H_SNAPSHOT_PATH))

In [ ]:
# =============================================================================
# Freeze the 12 H-identified STAR-to-upstream-input link candidates
# =============================================================================

upstream_links = []
for star in hits_h:
    output_id = star.get("file_id") or star.get("id")
    matches = idx_rna.loc[idx_rna["file_id"].eq(output_id)]
    if len(matches) != 1:
        raise ValueError("H STAR file is absent or duplicated in 111")
    row = matches.iloc[0]
    if (
        row["mapping_status"] != "unique_aliquot_sample"
        or row["sample_type"] != "Primary Tumor"
        or str(row["filename"]) != str(star.get("file_name"))
        or int(row["size_bytes"]) != int(star.get("file_size") or -1)
        or str(row["provider_md5"]).lower()
        != str(star.get("md5sum")).lower()
    ):
        raise ValueError("H STAR source identity/candidate status contradicted by 111")
    direct = star.get("associated_entities")
    if not isinstance(direct, list) or len(direct) != 1:
        raise ValueError("H output STAR direct association unexpectedly incomplete")
    entity = direct[0]
    if not (
        entity.get("entity_type") == "aliquot"
        and entity.get("entity_id") == row["aliquot_uuid"]
        and entity.get("case_id") == row["case_uuid"]
    ):
        raise ValueError("H STAR direct case+aliquot disagrees with 111")

    analysis = star.get("analysis") or {}
    if not isinstance(analysis, dict):
        raise ValueError("H STAR workflow analysis has invalid structure")
    inputs = analysis.get("input_files")
    if not isinstance(inputs, list) or len(inputs) != 1:
        raise ValueError("H pilot expected exactly one workflow input per STAR output")
    item = inputs[0]
    if not isinstance(item, dict) or not item.get("file_id"):
        raise ValueError("H workflow input has no exact source file UUID")
    upstream_links.append({
        "star_file_id": output_id,
        "case_uuid": str(row["case_uuid"]),
        "sample_uuid": str(row["sample_uuid"]),
        "aliquot_uuid": str(row["aliquot_uuid"]),
        "primary_site": str(row["primary_site"]),
        "disease_type": str(row["disease_type"]),
        "workflow_analysis_id": analysis.get("analysis_id"),
        "workflow_type": analysis.get("workflow_type"),
        "input_file_id": item["file_id"],
        "input_embedded": item,
    })

if (
    len(upstream_links) != 12
    or len({x["case_uuid"] for x in upstream_links}) != 12
    or len({x["star_file_id"] for x in upstream_links}) != 12
):
    raise ValueError("H source case/file scope changed")
INPUT_UUIDS_112 = sorted({x["input_file_id"] for x in upstream_links})
if not 1 <= len(INPUT_UUIDS_112) <= 12:
    raise ValueError("112 pilot exceeds permitted source input UUID count")
# Some API projections omit analysis_id despite retaining the input-file
# relationship. Preserve missing identity as a source caveat, not a guessed ID.
n_with_workflow_analysis_id = sum(
    bool(x.get("workflow_analysis_id")) for x in upstream_links
)

print("H output STAR files:", len(upstream_links))
print("STAR workflow analysis IDs populated:", n_with_workflow_analysis_id, "/ 12")
print("Exact upstream GDC input UUIDs:", len(INPUT_UUIDS_112))
print("Repeated inputs, if any, remain shared source nodes — not independent replicates")

In [ ]:
# =============================================================================
# A single bounded GDC /files metadata query with immutable exact-ID snapshot
# =============================================================================

GDC_FILES_URL = "https://api.gdc.cancer.gov/files"
FILES_FIELDS_112 = [
    "file_id", "file_name", "file_size", "md5sum", "access",
    "data_category", "data_type", "experimental_strategy",
    "total_reads", "proportion_reads_mapped",
    "proportion_reads_duplicated", "average_base_quality",
    "average_read_length", "average_insert_size",
    "analysis.analysis_id", "analysis.workflow_type",
    "analysis.workflow_version",
]
FILES_QUERY_112 = {
    "filters": {
        "op": "in",
        "content": {"field": "file_id", "value": INPUT_UUIDS_112},
    },
    "fields": ",".join(FILES_FIELDS_112),
    "expand": "associated_entities,analysis",
    "format": "JSON",
    "size": len(INPUT_UUIDS_112),
}
QUERY_DIGEST_112 = hashlib.sha256(json.dumps(
    FILES_QUERY_112, sort_keys=True, separators=(",", ":")
).encode("utf-8")).hexdigest()
SOURCE_HASH_111 = calculate_sha256(AUDIT_111_PATH)
SOURCE_HASH_H = calculate_sha256(H_SNAPSHOT_PATH)

if SNAPSHOT_112_PATH.is_file():
    with SNAPSHOT_112_PATH.open("r", encoding="utf-8") as h:
        snapshot_112 = json.load(h)
    if (
        snapshot_112.get("query_sha256") != QUERY_DIGEST_112
        or snapshot_112.get("upstream_111_summary_sha256") != SOURCE_HASH_111
        or snapshot_112.get("upstream_H_snapshot_sha256") != SOURCE_HASH_H
    ):
        raise ValueError("Existing 112 snapshot does not match input identity")
    print("Reusing immutable-on-rerun 112 upstream-source metadata snapshot")
else:
    # Source query returns *metadata only*. Missing records are recorded, not
    # misrepresented as provider-wide absence of a controlled read dataset.
    response = None
    for attempt in range(3):
        try:
            with requests.Session() as session:
                response = session.post(
                    GDC_FILES_URL, json=FILES_QUERY_112,
                    timeout=(20, 90),
                )
                response.raise_for_status()
            break
        except requests.RequestException as e:
            if attempt == 2:
                raise RuntimeError(
                    "GDC metadata request failed; DO NOT infer absent QC"
                ) from e
            time.sleep(2 ** attempt)
    if response is None:
        raise RuntimeError("No GDC API response")
    payload = response.json()
    direct_hits = (payload.get("data") or {}).get("hits")
    if not isinstance(direct_hits, list):
        raise ValueError("Unexpected GDC API shape; stop without source claims")
    received_ids = [x.get("file_id") or x.get("id") for x in direct_hits]
    if (
        len(direct_hits) > len(INPUT_UUIDS_112)
        or len(received_ids) != len(set(received_ids))
        or not set(received_ids).issubset(set(INPUT_UUIDS_112))
    ):
        raise ValueError("GDC API returned duplicate or unrequested input file IDs")
    snapshot_112 = {
        "notebook": 112,
        "role": "PHASE1_EXACT_UPSTREAM_QC_METADATA_PROVENANCE_ONLY",
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
        "url": GDC_FILES_URL,
        "query": FILES_QUERY_112,
        "query_sha256": QUERY_DIGEST_112,
        "upstream_111_summary_sha256": SOURCE_HASH_111,
        "upstream_H_snapshot_sha256": SOURCE_HASH_H,
        "hits": sorted(direct_hits, key=lambda x: x.get("file_id") or x.get("id")),
        "requested_input_file_ids": INPUT_UUIDS_112,
        "no_sequence_payloads_read_or_downloaded": True,
        "cohort_frozen": False,
    }
    SNAPSHOT_112_PATH.parent.mkdir(parents=True, exist_ok=True)
    with SNAPSHOT_112_PATH.open("x", encoding="utf-8") as h:
        json.dump(snapshot_112, h, indent=2, ensure_ascii=False)
        h.write("\n")
    print("Saved immutable-on-rerun 112 GDC input-record metadata snapshot")

direct_hits_112 = snapshot_112.get("hits")
returned_112 = {
    x.get("file_id") or x.get("id"): x
    for x in direct_hits_112
}
if (
    not isinstance(direct_hits_112, list)
    or len(returned_112) != len(direct_hits_112)
    or not set(returned_112).issubset(set(INPUT_UUIDS_112))
):
    raise ValueError("112 snapshot file ID subset/repetition mismatch")

print("Requested upstream input IDs:", len(INPUT_UUIDS_112))
print("Returned direct GDC file records:", len(returned_112))
print("Absent from exact source query:", len(set(INPUT_UUIDS_112) - set(returned_112)))
print("112 snapshot SHA-256:", calculate_sha256(SNAPSHOT_112_PATH))
print("No input BAM, FASTQ or STAR Count payload requested or read")

In [ ]:
# =============================================================================
# Link each upstream input file directly to the previously audited STAR aliquot
# =============================================================================

provenance_rows_112 = []
for rel in upstream_links:
    direct_input = returned_112.get(rel["input_file_id"])
    if direct_input is None:
        parent_status = "INPUT_RECORD_NOT_RETURNED_BY_EXACT_QUERY"
        records = None
        n_direct = 0
    else:
        records = direct_input.get("associated_entities")
        if not isinstance(records, list):
            parent_status = "INPUT_DIRECT_ASSOCIATIONS_NOT_EXPOSED"
            n_direct = 0
        else:
            n_direct = len(records)
            parents = [
                e for e in records if isinstance(e, dict)
                and e.get("entity_type") == "aliquot"
                and e.get("case_id") == rel["case_uuid"]
                and e.get("entity_id") == rel["aliquot_uuid"]
            ]
            if len(parents) == n_direct == 1:
                parent_status = "EXACT_INPUT_STAR_CASE_ALIQUOT_MATCH"
            else:
                parent_status = "MISMATCH_OR_NONUNIQUE_INPUT_ASSOCIATION"
    h_input = rel["input_embedded"]
    if direct_input is None:
        input_identity_consistency = "DIRECT_INPUT_FILE_RECORD_UNAVAILABLE"
    elif (
        h_input.get("file_name")
        and direct_input.get("file_name")
        and h_input["file_name"] != direct_input["file_name"]
    ):
        input_identity_consistency = "INPUT_FILENAME_CONTRADICTION"
    elif (
        h_input.get("access")
        and direct_input.get("access")
        and h_input["access"] != direct_input["access"]
    ):
        input_identity_consistency = "INPUT_ACCESS_STATUS_DIFFERENCE"
    else:
        input_identity_consistency = "NO_OBSERVED_INPUT_METADATA_CONTRADICTION"

    provenance_rows_112.append({
        "star_file_id": rel["star_file_id"],
        "case_uuid": rel["case_uuid"],
        "sample_uuid": rel["sample_uuid"],
        "aliquot_uuid": rel["aliquot_uuid"],
        "primary_site": rel["primary_site"],
        "disease_type": rel["disease_type"],
        "star_workflow_analysis_id": rel["workflow_analysis_id"],
        "star_workflow_type": rel["workflow_type"],
        "input_file_id": rel["input_file_id"],
        "input_direct_file_record_returned": direct_input is not None,
        "input_direct_case_aliquot_status": parent_status,
        "n_direct_entities_input": n_direct,
        "h_input_file_name": h_input.get("file_name"),
        "direct_input_file_name": (
            direct_input.get("file_name") if direct_input else None
        ),
        "h_input_data_type": h_input.get("data_type"),
        "direct_input_data_type": (
            direct_input.get("data_type") if direct_input else None
        ),
        "direct_input_access": (
            direct_input.get("access") if direct_input else None
        ),
        "input_file_metadata_consistency": input_identity_consistency,
        "direct_input_record_is_independent_quality_measurement": False,
        "assay_qc_denominators_validated": False,
        "analytic_sample_eligibility": "NOT_FROZEN",
    })

input_provenance_112 = pd.DataFrame(provenance_rows_112).sort_values(
    ["primary_site", "case_uuid", "star_file_id"], kind="stable"
).reset_index(drop=True)
if (
    len(input_provenance_112) != 12
    or input_provenance_112["star_file_id"].duplicated().any()
    or input_provenance_112["analytic_sample_eligibility"].ne(
        "NOT_FROZEN"
    ).any()
):
    raise ValueError("112 exact STAR-to-input identity scope changed")

display(input_provenance_112.groupby(
    ["primary_site", "input_direct_case_aliquot_status"],
    dropna=False
).agg(
    n_star_files=("star_file_id", "nunique")
).reset_index())
print("Case identity alone never licenses assigning upstream QC to STAR aliquot")

In [ ]:
# =============================================================================
# Compare two GDC metadata projections without calling them independent QC
# =============================================================================

def finite_numeric(value):
    if value is None or isinstance(value, (bool, list, dict)):
        return None
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return number if math.isfinite(number) else None


metric_rows_112 = []
for rel in upstream_links:
    live = returned_112.get(rel["input_file_id"])
    old = rel["input_embedded"]
    for name in QC_FIELDS:
        old_n = finite_numeric(old.get(name))
        live_n = finite_numeric(live.get(name)) if live else None
        if old_n is None:
            status = "H_FIELD_NOT_NUMERIC"
        elif live is None:
            status = "DIRECT_FILE_RECORD_UNAVAILABLE"
        elif live_n is None:
            status = "DIRECT_FILE_METRIC_NOT_NUMERIC_OR_NOT_RETURNED"
        elif math.isclose(old_n, live_n, rel_tol=1e-9, abs_tol=1e-9):
            status = "AGREES_ACROSS_PROVIDER_PROJECTIONS"
        else:
            status = "VALUE_DRIFT_BETWEEN_PROVIDER_PROJECTIONS"
        metric_rows_112.append({
            "star_file_id": rel["star_file_id"],
            "input_file_id": rel["input_file_id"],
            "case_uuid": rel["case_uuid"],
            "primary_site": rel["primary_site"],
            "metric_id": name,
            "H_input_embedded_value": old_n,
            "direct_input_file_value": live_n,
            "provider_projections_agreement": status,
            "independent_measurement_proven": False,
            "qc_rate_denominator_and_tool_version_verified": False,
            "same_library_quality_comparability_verified": False,
            "qc_pass_assigned": False,
        })

metric_comparison_112 = pd.DataFrame(metric_rows_112).sort_values(
    ["star_file_id", "metric_id"], kind="stable"
).reset_index(drop=True)
if (
    len(metric_comparison_112) != 12 * len(QC_FIELDS)
    or metric_comparison_112.duplicated(
        ["star_file_id", "metric_id"]
    ).any()
    or metric_comparison_112["qc_pass_assigned"].any()
):
    raise ValueError("112 metrics violated availability/non-QC-policy scope")

display(metric_comparison_112.groupby(
    ["metric_id", "provider_projections_agreement"]
).size().rename("n_star_files").reset_index())
print("Agreement is metadata consistency; never independent measurement")
print("No mapping-rate thresholds, QC PASS/FAIL labels or biological scores")

In [ ]:
# =============================================================================
# Classify feasibility with explicit remaining methodological gates
# =============================================================================

n_exact_input_links = int(
    input_provenance_112["input_direct_case_aliquot_status"].eq(
        "EXACT_INPUT_STAR_CASE_ALIQUOT_MATCH"
    ).sum()
)
n_same_provider_metrics = int(
    metric_comparison_112["provider_projections_agreement"].eq(
        "AGREES_ACROSS_PROVIDER_PROJECTIONS"
    ).sum()
)
n_direct_records = int(
    input_provenance_112["input_direct_file_record_returned"].sum()
)
if n_exact_input_links == 12 and n_same_provider_metrics == 72:
    source_feasibility_112 = "EXACT_SOURCE_LINK_AND_PROVIDER_METRICS_RECONCILED"
elif n_direct_records == 0:
    source_feasibility_112 = "NO_PUBLIC_DIRECT_INPUT_RECORDS_IN_BOUNDED_QUERY"
else:
    source_feasibility_112 = "PARTIAL_OR_DISCORDANT_SOURCE_LINKAGE"

# Matching the exact input aliquot resolves a source-provenance ambiguity, but
# GDC's aligned-read quality fields still require their own protocol and
# measurement-denominator documentation before use as QC eligibility cutoffs.
source_summary_112 = pd.DataFrame([
    {"metric": "H_source_STAR_output_files", "value": len(upstream_links)},
    {"metric": "distinct_upstream_input_file_ids", "value": len(INPUT_UUIDS_112)},
    {"metric": "GDC_input_direct_file_records", "value": n_direct_records},
    {"metric": "exact_direct_case_aliquot_input_links", "value": n_exact_input_links},
    {"metric": "provider_metric_values_agree", "value": n_same_provider_metrics},
    {"metric": "total_probe_metric_rows", "value": len(metric_comparison_112)},
    {"metric": "sequencing_qc_pass_conferred", "value": 0},
    {"metric": "RNA_samples_selected_or_excluded", "value": 0},
    {"metric": "QC_cutoffs_frozen", "value": 0},
    {"metric": "cohort_frozen", "value": 0},
])
if (
    source_summary_112.loc[
        source_summary_112["metric"].eq("H_source_STAR_output_files"),
        "value"
    ].iloc[0] != 12
    or input_provenance_112[
        "direct_input_record_is_independent_quality_measurement"
    ].any()
):
    raise ValueError("Source feasibility conflated with independent QC")

print("112 source-link feasibility:", source_feasibility_112)
print("Independent QC definition, denominator, assay/library version: NOT VERIFIED")
print("Automatically expand to other files: NO")
print("Regulatory/biological QC acceptance, cohort/lineage/score selection: NO")

In [ ]:
# =============================================================================
# Persist non-frozen Phase 1 source-QC handoff with exact input/output hashes
# =============================================================================

OUTPUT_112_PATHS["source_identities"].parent.mkdir(
    parents=True, exist_ok=True
)
tables_112 = {
    "source_identities": input_provenance_112,
    "metric_comparison": metric_comparison_112,
    "source_summary": source_summary_112,
}
for key, table in tables_112.items():
    table.to_csv(OUTPUT_112_PATHS[key], index=False)
    checked = pd.read_csv(OUTPUT_112_PATHS[key], dtype="string")
    if (
        len(checked) != len(table)
        or list(checked.columns) != list(table.columns)
    ):
        raise ValueError(f"112 persisted {key} does not match table contract")

handoff_112 = {
    "notebook": 112,
    "role": "PHASE1_UPSTREAM_RNA_QC_METADATA_PROVENANCE_PILOT",
    "status": "BOUNDED_SOURCE_FEASIBILITY_ONLY_NOT_QUALITY_QC_PASS",
    "source_feasibility": source_feasibility_112,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "prior_111_summary": {
        "path": project_relative_path(AUDIT_111_PATH),
        "sha256": SOURCE_HASH_111,
    },
    "prior_111_file_index": {
        "path": project_relative_path(FILE_INDEX_111_PATH),
        "sha256": calculate_sha256(FILE_INDEX_111_PATH),
    },
    "prior_H_snapshot": {
        "path": project_relative_path(H_SNAPSHOT_PATH),
        "sha256": SOURCE_HASH_H,
        "query_sha256": snapshot_h["query_sha256"],
    },
    "input_direct_record_snapshot": {
        "path": project_relative_path(SNAPSHOT_112_PATH),
        "sha256": calculate_sha256(SNAPSHOT_112_PATH),
        "query_sha256": QUERY_DIGEST_112,
        "captured_at_utc": snapshot_112.get("captured_at_utc"),
    },
    "observations": {
        "H_STAR_output_files": len(upstream_links),
        "unique_upstream_input_IDs_requested": len(INPUT_UUIDS_112),
        "upstream_input_direct_records_returned": n_direct_records,
        "exact_direct_case_and_aliquot_links": n_exact_input_links,
        "agreeing_provider_metric_field_pairs": n_same_provider_metrics,
        "metric_field_pairs_total": len(metric_comparison_112),
    },
    "unresolved_gates": [
        "provider_field_tool_version_and_denominator_definitions",
        "comparable_library_and_assay_measurement_scope",
        "full_CPTAC3_QC_coverage_and_missingness_not_assessed",
        "lineage_and_sample_specific_QC_sensitivity_not_frozen",
        "cross_cohort_donor_independence_not_assessed_in_112",
        "RNA_case_multiplicity_policy_not_assessed_in_112",
    ],
    "data_access_scope": "PUBLIC_GDC_FILE_METADATA_ONLY",
    "payload_downloads": 0,
    "large_STAR_scans": 0,
    "automatic_API_expansion": False,
    "QC_cutoffs_frozen": 0,
    "samples_selected_or_excluded": 0,
    "cohort_frozen": False,
    "program_scores_computed": False,
    "biological_replication_performed": False,
    "ready_for_800_review": True,
    "QC_eligibility_authorized": False,
    "outputs": {
        key: {
            "path": project_relative_path(OUTPUT_112_PATHS[key]),
            "rows": len(table),
            "sha256": calculate_sha256(OUTPUT_112_PATHS[key]),
        } for key, table in tables_112.items()
    },
}
with OUTPUT_112_PATHS["handoff"].open("w", encoding="utf-8") as h:
    json.dump(handoff_112, h, indent=2, ensure_ascii=False)
    h.write("\n")
with OUTPUT_112_PATHS["handoff"].open("r", encoding="utf-8") as h:
    persisted = json.load(h)
if (
    persisted["observations"] != handoff_112["observations"]
    or persisted["QC_eligibility_authorized"]
    or persisted["cohort_frozen"]
):
    raise ValueError("112 handoff persistence/eligibility boundary failed")

for key, table in tables_112.items():
    print(key, "→", project_relative_path(OUTPUT_112_PATHS[key]), len(table))
print("Handoff:", project_relative_path(OUTPUT_112_PATHS["handoff"]))
print("Original 111/H snapshots untouched; Phase 8 eligibility still NOT FROZEN")

## Prospective closure interpretation — pending first local execution

**What a successful Run All will establish:** exactly scoped acquisition/audit evidence for the same 12 GDC workflow-input file UUIDs identified in H, comparing independent *file-record attribution* with its parent STAR output's exact case+aliquot identity. It will create an immutable source query and a versioned non-frozen 112 handoff for subsequent inspection.

A complete same-case/same-aliquot match is **stronger provenance linkage**, not evidence that two GDC metadata projections provide independently estimated read QC. Likewise, six populated aligned-read metrics do not by themselves establish provider calculation methods, library equivalence, an appropriate mapped-read denominator, quality acceptability, or validated sample eligibility. Zero/partial returned records and absent direct associations are reportable negative feasibility outcomes without guessing new source links.

**STOP after this 12-output pilot.** Do not scale public API requests to 1,082 candidate STAR files or 649 multiplicity files until an explicitly reviewed Phase 1 acquisition decision; do not promote any primary candidate or change the Phase 8 cohort. If a source can be audited reliably, the separate 800 contract must still predeclare a lineage-aware QC missingness policy, 288-case multiplicity handling, TCGA–CPTAC donor independence, and a matched-null replication estimand.

**Handoff is source evidence, not a frozen biological analysis:** downstream Notebook 800 may cite the accepted 112 source metadata and limitations after technical review without rerunning 112, querying GDC in its own analysis, or reopening Notebook 111. The original H snapshot and 111 acquisition outputs remain unchanged.